# Full Experiment Run

This notebook runs the time-aware RecBole -> carbon-aware reranking -> paper-plot pipeline into `run/`, matching the methodology in `docs/main.tex`.

It is designed for Google Colab, but it also works in a local Jupyter environment if the repo is already checked out.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

In [ ]:
from pathlib import Path

candidates = [
    Path('/content/carbon-aware-recsys'),
    Path('/content/drive/MyDrive/carbon-aware-recsys'),
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
]
PROJECT_ROOT = next(
    (path for path in candidates if (path / 'scripts' / '05_run_full_experiment.py').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Set PROJECT_ROOT to the repo root before running the rest of the notebook.'
    )
RUN_DIR = PROJECT_ROOT / 'run'
PROJECT_ROOT, RUN_DIR

In [ ]:
%cd {PROJECT_ROOT}
%pip install -q -r requirements.txt

In [ ]:
CATEGORIES = ['electronics', 'home_and_kitchen', 'sports_and_outdoors']
MODELS = ['BPR', 'NeuMF', 'LightGCN']

TOP_K_CANDIDATES = 100
TOP_K_RERANK = 10
USER_BATCH_SIZE = 256
TRAIN_BATCH_SIZE = 2048
EVAL_BATCH_SIZE = 4096
LEARNING_RATE = 1e-3
EPOCHS = 50

MAX_PARALLEL_JOBS = None
MAX_USERS = None  # Set to a small integer like 5000 for a quick smoke run.

PREPARE_CARBON = False
FORCE = False
FORCE_CARBON = False
SKIP_LLM = False
LLM_CACHE_ONLY = False

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    'scripts/05_run_full_experiment.py',
    '--run-dir', str(RUN_DIR),
    '--interim-dir', str(PROJECT_ROOT / 'data' / 'interim'),
    '--top-k-candidates', str(TOP_K_CANDIDATES),
    '--top-k-rerank', str(TOP_K_RERANK),
    '--user-batch-size', str(USER_BATCH_SIZE),
    '--train-batch-size', str(TRAIN_BATCH_SIZE),
    '--eval-batch-size', str(EVAL_BATCH_SIZE),
    '--learning-rate', str(LEARNING_RATE),
    '--epochs', str(EPOCHS),
    '--categories', *CATEGORIES,
    '--models', *MODELS,
]
if MAX_PARALLEL_JOBS is not None:
    cmd.extend(['--max-parallel-jobs', str(MAX_PARALLEL_JOBS)])
if MAX_USERS is not None:
    cmd.extend(['--max-users', str(MAX_USERS)])
if PREPARE_CARBON:
    cmd.append('--prepare-carbon')
if FORCE:
    cmd.append('--force')
if FORCE_CARBON:
    cmd.append('--force-carbon')
if SKIP_LLM:
    cmd.append('--skip-llm')
if LLM_CACHE_ONLY:
    cmd.append('--llm-cache-only')

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

summary_path = RUN_DIR / 'results' / 'dataset_summary.csv'
best_points_path = RUN_DIR / 'results' / 'paper_best_pareto_points.csv'
manifest_path = RUN_DIR / 'results' / 'paper_plot_manifest.json'

if summary_path.exists():
    display(pd.read_csv(summary_path))
if best_points_path.exists():
    display(pd.read_csv(best_points_path))
if manifest_path.exists():
    with manifest_path.open() as handle:
        manifest = json.load(handle)
    print(f"Generated {len(manifest['figures'])} figures")
    for figure_path in manifest['figures'][:6]:
        display(Image(filename=figure_path))